# Derco Chile — Vehicle Sales 2009-2022
## Initial Exploration & BI Scoping

---

**Author:** BI Team  
**Dataset:** `ventas_derco_encrypt_exam_TOPD_2009_2022.xlsx`  
**Purpose:** First-pass exploration of Derco Chile's historical retail vehicle-sales dataset. The goal of this notebook is **not** to produce final metrics, but to (1) understand what the data actually contains, (2) surface data-quality risks, and (3) propose a shortlist of **BI domains** the team can develop in follow-up work.

### Business context
Derco is one of the largest multi-brand vehicle importers and dealers in Chile. The file we received covers **14 years of retail transactions** across multiple brands and channels. The data was anonymized (customer RUT is hashed) and list prices are synthetic — margins/prices should therefore be treated as **relative signals**, not literal financial figures.

### Notebook structure
1. **Introduction** — libraries, loading, data dictionary  
2. **Exploratory Data Analysis (EDA)** — quality, temporal, brand, geography, customer  
3. **Candidate BI domains** — 4 workstreams the team can prioritize next

---
## 1. Introduction

### 1.1 Libraries
Standard Python analytical stack. `plotly` is used for interactive charts intended for dashboards; `matplotlib` for static exploratory plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('pandas', pd.__version__)

### 1.2 Loading the data

The source file is an `.xlsx` with a single sheet (`in`) whose one column contains all fields **pipe-delimited** (`|`). We split it into a proper tabular structure before any analysis.

In [ ]:
SRC = '../data/ventas_derco_encrypt_exam_TOPD_2009_2022.xlsx'

raw = pd.read_excel(SRC, sheet_name='in')
print('Raw shape:', raw.shape)
print('Raw column header:', raw.columns.tolist()[0])

In [ ]:
COLS = [
    'fecha_transaccion',
    'encrypt_rut',
    'direccion',
    'comuna',
    'marca',
    'detalle',
    'margen_retail',
    'retail',
    'year',
    'precio_de_lista_synt',
]

df = raw.iloc[:, 0].astype(str).str.split('|', expand=True)
df.columns = COLS

df['fecha_transaccion'] = pd.to_datetime(df['fecha_transaccion'], errors='coerce')
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df['margen_retail'] = pd.to_numeric(df['margen_retail'], errors='coerce')
df['precio_de_lista_synt'] = pd.to_numeric(df['precio_de_lista_synt'], errors='coerce')

for c in ['direccion', 'comuna', 'marca', 'detalle', 'retail']:
    df[c] = df[c].replace({'': np.nan, 'nan': np.nan}).str.strip()

df.head()

### 1.3 Data dictionary

| Field | Type | Description |
|---|---|---|
| `fecha_transaccion` | date | Date the vehicle sale was booked |
| `encrypt_rut` | string (hash) | Anonymized customer ID (SHA-style) — use as customer key |
| `direccion` | string | Customer street address (free text) |
| `comuna` | string | Chilean municipality (~geographic unit) |
| `marca` | string | Vehicle brand |
| `detalle` | string | Model + trim descriptor |
| `margen_retail` | float | Retail margin per transaction (CLP, synthetic) |
| `retail` | string | Sales channel — `ces` (dealer network) vs `propio` (owned store) |
| `year` | int | Transaction year (redundant with `fecha_transaccion`) |
| `precio_de_lista_synt` | float | Synthetic list price (CLP) |

> ⚠️ **Interpretation caveat:** `precio_de_lista_synt` and `margen_retail` are synthetically generated / obfuscated for the exam dataset. Absolute money values are **not** production figures; ratios and rankings are still analytically valid.

---
## 2. Exploratory Data Analysis

### 2.1 Shape & data quality

In [ ]:
print(f'Rows: {len(df):,}')
print(f'Columns: {df.shape[1]}')
print(f'Date range: {df["fecha_transaccion"].min().date()} → {df["fecha_transaccion"].max().date()}')
print(f'Unique customers (encrypt_rut): {df["encrypt_rut"].nunique():,}')
print(f'Unique brands: {df["marca"].nunique()}')
print(f'Unique models (detalle): {df["detalle"].nunique():,}')
print(f'Unique comunas: {df["comuna"].nunique()}')

In [ ]:
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'nulls': df.isna().sum(),
    'null_pct': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique(),
})
quality

**Observations**
- Missing values are concentrated on `direccion` / `comuna` (~0.7%). These records still carry valid transactional data — we should **flag rather than drop** so we don't distort revenue totals.
- `year` is redundant with `fecha_transaccion`. Keep only one to avoid consistency drift.
- `encrypt_rut` uniqueness (~464K over ~550K rows) implies a ~1.18 transactions-per-customer average → **very low repeat rate** on the surface, worth a deeper look in the Customer domain.

In [ ]:
# Sanity: fecha_transaccion.year should match `year` column
mismatch = (df['fecha_transaccion'].dt.year != df['year']).sum()
print(f'Rows where fecha.year != year column: {mismatch}')

# Duplicate row check
dup = df.duplicated().sum()
print(f'Exact duplicate rows: {dup}')

### 2.2 Univariate distributions

#### Brands (marca)

In [ ]:
brand_counts = df['marca'].value_counts()
brand_share = (brand_counts / brand_counts.sum() * 100).round(2)
brand_summary = pd.DataFrame({'transactions': brand_counts, 'share_%': brand_share})
brand_summary

In [ ]:
ax = brand_counts.plot(kind='barh', color='steelblue')
ax.invert_yaxis()
ax.set_title('Transactions by brand (2009-2022)')
ax.set_xlabel('Number of transactions')
plt.tight_layout()
plt.show()

#### Retail channel

In [ ]:
channel = df['retail'].value_counts(dropna=False)
print(channel)
channel.plot(kind='pie', autopct='%1.1f%%', ylabel='', title='Sales channel mix')
plt.show()

#### Price & margin distributions

In [ ]:
df[['precio_de_lista_synt', 'margen_retail']].describe().round(0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df['precio_de_lista_synt'].plot(kind='hist', bins=60, ax=axes[0], color='steelblue')
axes[0].set_title('List price (synthetic) — distribution')
axes[0].set_xlabel('CLP')

df['margen_retail'].plot(kind='hist', bins=60, ax=axes[1], color='seagreen')
axes[1].set_title('Retail margin — distribution')
axes[1].set_xlabel('CLP')
plt.tight_layout()
plt.show()

### 2.3 Temporal patterns

How did volume and revenue evolve across the 14-year window?

In [ ]:
df['year_month'] = df['fecha_transaccion'].dt.to_period('M').dt.to_timestamp()

monthly = df.groupby('year_month').agg(
    transactions=('encrypt_rut', 'size'),
    revenue=('precio_de_lista_synt', 'sum'),
    margin=('margen_retail', 'sum'),
)
monthly.head()

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 5))
ax1.plot(monthly.index, monthly['transactions'], color='steelblue', label='Transactions')
ax1.set_ylabel('Transactions / month', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly['revenue'] / 1e9, color='darkorange', label='Revenue (bn CLP)')
ax2.set_ylabel('Revenue (bn CLP, synthetic)', color='darkorange')
ax2.tick_params(axis='y', labelcolor='darkorange')
ax2.grid(False)

plt.title('Monthly volume vs revenue — Derco Chile, 2009-2022')
plt.tight_layout()
plt.show()

In [ ]:
# Seasonality view — average transactions per calendar month
df['month'] = df['fecha_transaccion'].dt.month
seasonal = df.groupby('month').size()
seasonal.plot(kind='bar', color='steelblue')
plt.title('Seasonality — total transactions per calendar month (all years pooled)')
plt.xlabel('Month')
plt.ylabel('Transactions')
plt.xticks(range(12), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=0)
plt.tight_layout()
plt.show()

### 2.4 Brand evolution — market share over time

In [ ]:
brand_year = df.groupby(['year', 'marca']).size().unstack(fill_value=0)
brand_share_yr = brand_year.div(brand_year.sum(axis=1), axis=0) * 100

brand_share_yr.plot(kind='area', stacked=True, colormap='tab20', figsize=(13, 5))
plt.title('Brand share of transactions, by year (%)')
plt.ylabel('Share (%)')
plt.xlabel('Year')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

### 2.5 Geography — where do Derco's customers live?

In [ ]:
top_comunas = (
    df.groupby('comuna')
      .agg(transactions=('encrypt_rut', 'size'),
           revenue=('precio_de_lista_synt', 'sum'))
      .sort_values('transactions', ascending=False)
      .head(20)
)
top_comunas

In [ ]:
top_comunas['transactions'].sort_values().plot(kind='barh', color='steelblue', figsize=(9, 7))
plt.title('Top 20 comunas by transaction volume')
plt.xlabel('Transactions')
plt.tight_layout()
plt.show()

In [ ]:
# Concentration: what share of sales lives in the top-N comunas?
comuna_rank = df['comuna'].value_counts()
cum_share = comuna_rank.cumsum() / comuna_rank.sum() * 100
print(f'Top 10 comunas cover: {cum_share.iloc[9]:.1f}% of transactions')
print(f'Top 50 comunas cover: {cum_share.iloc[49]:.1f}% of transactions')
print(f'Total comunas with at least one sale: {len(comuna_rank)}')

### 2.6 Customer behavior — repeat purchase

In [ ]:
cust_freq = df.groupby('encrypt_rut').size()
freq_dist = cust_freq.value_counts().sort_index()
print('Distribution of purchases per customer:')
print(freq_dist.head(10))
print()
print(f'One-time buyers: {(cust_freq == 1).sum():,} ({(cust_freq == 1).mean()*100:.1f}%)')
print(f'Repeat buyers (2+): {(cust_freq >= 2).sum():,} ({(cust_freq >= 2).mean()*100:.1f}%)')
print(f'Max purchases by a single customer: {cust_freq.max()}')

### 2.7 EDA — key takeaways

- **Volume is dominated by Suzuki** (~39% of transactions), followed by Mazda and Renault. Chinese brands (JAC, Great Wall, Changan, Geely, Haval) collectively represent a growing block worth isolating in trend analysis.
- **Dealer network (`ces`) drives ~65% of sales**, own stores the rest. Channel productivity is a natural KPI.
- **Geographic concentration is high** — a small subset of comunas explains a large share of volume, which points at both a strength (density) and a risk (over-exposure).
- **Repeat purchase rate is low** — most customers appear only once in 14 years. Either the vehicle-purchase cycle genuinely is that long, or we're missing post-sale touchpoints. Either interpretation is a BI opportunity.
- **Data-quality is broadly clean**: <1% nulls, no exact duplicates, `year` consistent with `fecha_transaccion`. Safe to build downstream models on top.

---
## 3. Candidate BI Domains

Four workstreams the team can develop next. Each is scoped so that a first dashboard / model could ship within one sprint.

### 🟦 Domain 1 — Sales Performance & Forecasting
**Question we answer:** *How are we selling, and what should we expect next quarter?*

- KPIs: monthly revenue, YoY growth, run-rate vs plan, seasonality index.
- Deliverables: executive sales dashboard (revenue, margin, units) + a monthly demand-forecast model (SARIMA / Prophet baseline) at brand level.
- Why this dataset supports it: 14 years of continuous monthly signal, clean dates, revenue and margin per line.
- Watch-outs: the pandemic period (2020) breaks stationarity — the model needs a COVID regime flag.

### 🟩 Domain 2 — Brand & Product Portfolio Analytics
**Question we answer:** *Which brands and models pull their weight, and where is portfolio risk?*

- KPIs: brand market share evolution, top-N model contribution, ABC/Pareto on `detalle`, price-band mix, margin-per-model ranking.
- Deliverables: portfolio scorecard (BCG-style growth vs share matrix) at brand and model level; alerting on models with declining share year-over-year.
- Why this dataset supports it: 1,543 unique models tagged with brand, list price and margin — enough granularity for portfolio segmentation.
- Watch-outs: `precio_de_lista_synt` is synthetic; use rankings, not absolute pricing.

### 🟨 Domain 3 — Geographic / Territory Intelligence
**Question we answer:** *Where do we sell, where should we sell more, and where are we over-exposed?*

- KPIs: sales density per comuna, growth vs national baseline, brand-mix per region, dealer coverage vs demand.
- Deliverables: choropleth of Chile at comuna level joined to demographic/INE population data → market-penetration index; a shortlist of under-served but high-potential comunas for expansion.
- Why this dataset supports it: 512 comunas covered, transactional address available, brand & channel dimensions to slice by.
- Watch-outs: comuna free-text needs a canonical mapping to INE codes before joining external population data.

### 🟥 Domain 4 — Customer & Channel Analytics
**Question we answer:** *Who is buying from us, and is the `ces` vs `propio` channel mix optimal?*

- KPIs: one-time vs repeat customer split, customer lifetime value (approximated by cumulative margin per `encrypt_rut`), channel-level margin per transaction, brand affinity per channel.
- Deliverables: customer segmentation (RFM-lite: Recency, Frequency, Monetary using synthetic price as proxy), channel performance dashboard, cross-sell opportunity list for the small population of repeat buyers.
- Why this dataset supports it: hashed but stable customer key + full transaction history + channel tag.
- Watch-outs: repeat rate is very low → validate with the business that the encryption key is stable across the full 14 years before drawing loyalty conclusions.

---
## Next steps

1. **Validate assumptions with the business owner** — confirm the meaning of `ces` vs `propio`, and whether `encrypt_rut` is stable across the full history.
2. **Prioritize one of the four domains** for the next iteration; recommend starting with **Domain 1 (Sales Performance)** as the foundational layer every other domain will re-use.
3. **Enrich with external data** — Chilean INE comuna master, brand-level industry sales (ANAC) for market-share benchmarking.
4. **Move raw file to a partitioned Parquet** in the data lake so downstream notebooks stop paying the ~30 s Excel parse cost every run.